In [1]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFECV


In [3]:
# Build a .py script that takes a snapshot date, trains a model and outputs artefact into storage.

## set up pyspark session

In [2]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/25 14:44:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## set up config

In [ ]:
import pprint
from datetime import datetime


# --- 1. Define the specific date ranges ---
config = {}

config["train_start_date"] = datetime(2023, 1, 1)
config["train_end_date"] = datetime(2023, 12, 1)

config["test_start_date"] = datetime(2024, 3, 1)
config["test_end_date"] = datetime(2024, 4, 1)

# Validation period 
config["val_start_date"] = datetime(2024, 1, 1)
config["val_end_date"] = datetime(2024, 2, 1)

# Out-of-Time (OOT) period
config["oot_start_date"] = datetime(2024, 5, 1)
config["oot_end_date"] = datetime(2024, 5, 1)

# --- 2. Print the final config ---
pprint.pprint(config)

{'oot_end_date': datetime.datetime(2024, 6, 30, 0, 0),
 'oot_start_date': datetime.datetime(2024, 6, 1, 0, 0),
 'train_test_end_date': datetime.datetime(2024, 5, 31, 0, 0),
 'train_test_ratio': 0.8,
 'train_test_start_date': datetime.datetime(2023, 1, 1, 0, 0),
 'val_end_date': datetime.datetime(2024, 3, 31, 0, 0),
 'val_start_date': datetime.datetime(2024, 1, 1, 0, 0)}


## get label store

In [46]:
# connect to label store
folder_path = "datamart/gold/label_store/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
label_store_sdf = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",label_store_sdf.count())

label_store_sdf.show()

row_count: 8974
+--------------------+-----------+-----+----------+-------------+
|             loan_id|customer_id|label| label_def|snapshot_date|
+--------------------+-----------+-----+----------+-------------+
|CUS_0x1037_2023_0...| CUS_0x1037|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1069_2023_0...| CUS_0x1069|    0|30dpd_6mob|   2023-07-01|
|CUS_0x114a_2023_0...| CUS_0x114a|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1184_2023_0...| CUS_0x1184|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1297_2023_0...| CUS_0x1297|    1|30dpd_6mob|   2023-07-01|
|CUS_0x12fb_2023_0...| CUS_0x12fb|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1325_2023_0...| CUS_0x1325|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1341_2023_0...| CUS_0x1341|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1375_2023_0...| CUS_0x1375|    1|30dpd_6mob|   2023-07-01|
|CUS_0x13a8_2023_0...| CUS_0x13a8|    0|30dpd_6mob|   2023-07-01|
|CUS_0x13ef_2023_0...| CUS_0x13ef|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1440_2023_0...| CUS_0x1440|    0|30dpd_6mob|   2023-0

In [67]:
# extract label store
labels_sdf = label_store_sdf

#.filter((col("snapshot_date") >= config["train_test_start_date"]) & (col("snapshot_date") <= config["oot_end_date"]))

print("extracted labels_sdf", labels_sdf.count())
      #, config["train_test_start_date"], config["oot_end_date"])

extracted labels_sdf 8974


## get features

In [48]:
# connect to feature store
folder_path = "datamart/gold/feature_store/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
feature_store_sdf = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",feature_store_sdf.count())

feature_store_sdf.show()

row_count: 8974
+-----------+-------------+---+-------------+---------------------+-----------------+---------------+-------------+-----------+-------------------+----------------------+--------------------+--------------------+----------------+------------------------+-------------------+-----------------------+---------------+------------------------+-------------+---------+-------------------+-------------+-------------+------------+----------------+-----------+-----------------------+-----------------+--------------------+-----------------+-------------------+------------------------+-------------------+--------------------+--------------------+------------------+---------------------+-----------------------+---------------------+-------------------+-----------------+------------------+-------------------------+------------------------+------------------------+-------------------+---------------+--------------+---------------------------+----------------------------+---------------

In [49]:
# extract feature store
features_sdf = feature_store_sdf.filter((col("snapshot_date") >= config["train_test_start_date"]) & (col("snapshot_date") <= config["oot_end_date"]))

print("extracted features_sdf", features_sdf.count(), config["train_test_start_date"], config["oot_end_date"])

[Stage 54:===>                                                    (1 + 14) / 15]

extracted features_sdf 8974 2023-01-01 00:00:00 2024-06-30 00:00:00


## prepare data for modeling

In [68]:
from pyspark.sql.functions import col, add_months

# 1. Perform the join using the explicit, two-part condition
# We are joining where IDs match AND the label date is 6 months after the feature date.
joined_sdf = labels_sdf.join(
    features_sdf,
    on=[
        labels_sdf.customer_id == features_sdf.customer_id,
        labels_sdf.snapshot_date == add_months(features_sdf.snapshot_date, 6)
    ],
    how="inner"
)

# 2. Drop the columns you don't want
# This creates a new DataFrame, final_sdf, that:
#   - Drops the snapshot_date from the 'labels_sdf' table
#   - Drops the redundant customer_id from the 'labels_sdf' table
# This leaves you with the customer_id and snapshot_date from 'features_sdf'.

final_sdf = joined_sdf.drop(labels_sdf.snapshot_date) \
                      .drop(features_sdf.customer_id)

# 3. Convert the final, clean DataFrame to Pandas
data_pdf = final_sdf.toPandas()

# 4. Check the result
print("Join successful. Final DataFrame head:")
print(data_pdf.head())

print("\nFinal DataFrame columns:")
print(data_pdf.columns)

Join successful. Final DataFrame head:
                 loan_id customer_id  label   label_def snapshot_date  age  \
0  CUS_0x10c0_2024_02_01  CUS_0x10c0      1  30dpd_6mob    2024-02-01   39   
1  CUS_0x12ef_2024_02_01  CUS_0x12ef      0  30dpd_6mob    2024-02-01   43   
2  CUS_0x1414_2024_02_01  CUS_0x1414      0  30dpd_6mob    2024-02-01   46   
3  CUS_0x18f4_2024_02_01  CUS_0x18f4      0  30dpd_6mob    2024-02-01   23   
4  CUS_0x1935_2024_02_01  CUS_0x1935      0  30dpd_6mob    2024-02-01   44   

   annual_income  monthly_inhand_salary  num_bank_accounts  num_credit_card  \
0   49454.128906            4328.177734                8.0              5.0   
1   68898.000000            5588.500000                7.0              4.0   
2   51093.121094            4080.760010                8.0              9.0   
3   35805.250000            3106.770752                4.0              7.0   
4   17096.250000            1669.687500               10.0              8.0   

   ...   avg_fe_1

In [71]:
data_pdf.tail()

,loan_id,customer_id,label,label_def,snapshot_date,age,annual_income,monthly_inhand_salary,num_bank_accounts,num_credit_card,...,avg_fe_11,avg_fe_12,avg_fe_13,avg_fe_14,avg_fe_15,avg_fe_16,avg_fe_17,avg_fe_18,avg_fe_19,avg_fe_20
8969,CUS_0xbfb4_2023_09_01,CUS_0xbfb4,0,30dpd_6mob,2023-09-01,37,17983.019531,1522.584961,5.0,6.0,...,113.333333,89.777778,123.000000,142.444444,33.111111,60.444444,112.666667,64.333333,132.555556,29.222222
8970,CUS_0xc242_2023_09_01,CUS_0xc242,0,30dpd_6mob,2023-09-01,18,17018.449219,1188.204224,8.0,6.0,...,85.111111,81.555556,94.888889,114.000000,70.888889,19.888889,51.555556,109.111111,121.111111,120.555556
8971,CUS_0xc26e_2023_09_01,CUS_0xc26e,0,30dpd_6mob,2023-09-01,42,52395.839844,4278.319824,6.0,3.0,...,61.777778,114.666667,81.888889,82.777778,119.444444,122.666667,97.111111,97.000000,121.777778,77.666667
8972,CUS_0xc2cb_2023_09_01,CUS_0xc2cb,0,30dpd_6mob,2023-09-01,33,22387.125000,1677.593750,4.0,3.0,...,89.333333,85.777778,122.666667,154.666667,92.444444,103.888889,129.333333,104.444444,77.777778,137.333333
8973,CUS_0xc610_2023_09_01,CUS_0xc610,1,30dpd_6mob,2023-09-01,33,56175.121094,4916.259766,5.0,7.0,...,99.777778,138.666667,77.777778,70.333333,88.222222,114.333333,42.222222,25.444444,91.000000,74.000000


In [70]:
# --- 1. Split data into time-based blocks ---

# OOT data (e.g., June 2024)
oot_pdf = data_pdf[
    (data_pdf['snapshot_date'] >= config["oot_start_date"].date()) & 
    (data_pdf['snapshot_date'] <= config["oot_end_date"].date())
]

# Validation data (e.g., Jan 2024 - Mar 2024)
val_pdf = data_pdf[
    (data_pdf['snapshot_date'] >= config["val_start_date"].date()) & 
    (data_pdf['snapshot_date'] <= config["val_end_date"].date())
]

# Get the full "train_test" block (e.g., 2023-01-01 to 2024-05-31)
train_test_full_pdf = data_pdf[
    (data_pdf['snapshot_date'] >= config["train_test_start_date"].date()) & 
    (data_pdf['snapshot_date'] <= config["train_test_end_date"].date())
]

# --- 2. Create the final train/test set by EXCLUDING validation data ---
# This is the crucial step: we remove the validation rows from the main block.

# Get the indices of rows that are *not* in the validation set
train_test_indices = ~train_test_full_pdf.index.isin(val_pdf.index)

# Create the final train_test_pdf (all data *except* val and oot)
train_test_pdf = train_test_full_pdf[train_test_indices]

# --- 3. Define feature columns ---
feature_cols = [col for col in features_sdf.columns if col not in ['customer_id', 'snapshot_date']]

# --- 4. Create X, y for OOT and Validation sets ---
X_oot = oot_pdf[feature_cols]
y_oot = oot_pdf["label"]

X_val = val_pdf[feature_cols]
y_val = val_pdf["label"]

# --- 5. Create X, y for Train and Test sets ---
# Now we split the remaining data (train_test_pdf) randomly.
X_train, X_test, y_train, y_test = train_test_split(
    train_test_pdf[feature_cols], 
    train_test_pdf["label"],
    test_size= 1 - config.get("train_test_ratio", 0.8), # Use .get for safety
    random_state=88,
    shuffle=True,
    stratify=train_test_pdf["label"]
)

# --- 6. Print summary ---
print('--- Data Split Summary ---')
print(f'X_train: {X_train.shape[0]} rows')
print(f'X_val:   {X_val.shape[0]} rows')
print(f'X_test:  {X_test.shape[0]} rows')
print(f'X_oot:   {X_oot.shape[0]} rows')

print('\n--- Label Distribution ---')
print(f'y_train: {y_train.shape[0]} rows, mean={round(y_train.mean(), 2)}')
print(f'y_val:   {y_val.shape[0]} rows, mean={round(y_val.mean(), 2)}')
print(f'y_test:  {y_test.shape[0]} rows, mean={round(y_test.mean(), 2)}')
print(f'y_oot:   {y_oot.shape[0]} rows, mean={round(y_oot.mean(), 2)}')

X_train

--- Data Split Summary ---
X_train: 5569 rows
X_val:   1514 rows
X_test:  1393 rows
X_oot:   498 rows

--- Label Distribution ---
y_train: 5569 rows, mean=0.28
y_val:   1514 rows, mean=0.3
y_test:  1393 rows, mean=0.28
y_oot:   498 rows, mean=0.31


,age,annual_income,monthly_inhand_salary,num_bank_accounts,num_credit_card,interest_rate,num_of_loan,delay_from_due_date,num_of_delayed_payment,changed_credit_limit,...,avg_fe_11,avg_fe_12,avg_fe_13,avg_fe_14,avg_fe_15,avg_fe_16,avg_fe_17,avg_fe_18,avg_fe_19,avg_fe_20
4170,46,93233.726562,7762.477539,6.0,7.0,17.0,7.0,11,14.0,16.51,...,96.000000,23.800000,133.800000,78.200000,62.600000,35.200000,75.800000,154.800000,75.400000,38.200000
7584,37,74036.960938,6384.746582,9.0,10.0,15.0,5.0,59,18.0,9.74,...,116.500000,73.250000,132.250000,173.750000,104.250000,107.250000,129.250000,199.500000,113.000000,31.500000
3821,35,33352.410156,2671.207520,6.0,4.0,10.0,2.0,13,22.0,17.68,...,106.083333,66.083333,74.833333,109.333333,167.833333,105.500000,63.000000,101.250000,129.000000,90.583333
5108,36,70913.398438,6126.450195,8.0,4.0,12.0,3.0,15,12.0,3.24,...,96.000000,110.800000,88.700000,75.200000,137.400000,64.500000,74.900000,101.300000,83.400000,98.500000
8792,17,59930.640625,4988.220215,6.0,7.0,33.0,4.0,61,20.0,13.43,...,128.625000,138.125000,114.875000,73.125000,77.250000,86.125000,69.000000,120.000000,55.875000,107.125000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
892,34,17346.160156,1607.513306,8.0,9.0,25.0,5.0,52,10.0,10.45,...,124.666667,104.777778,127.888889,119.222222,86.222222,77.111111,92.555556,68.000000,72.333333,67.333333
3461,31,33445.011719,2663.084229,6.0,4.0,9.0,0.0,8,12.0,4.82,...,132.400000,78.600000,87.600000,87.600000,121.200000,32.200000,109.400000,57.400000,90.800000,81.400000
931,22,122566.593750,10323.882812,0.0,7.0,1.0,0.0,18,10.0,1.14,...,77.222222,133.222222,76.333333,118.222222,115.222222,143.555556,118.666667,91.333333,96.777778,77.111111
7296,42,8856.750000,989.062500,6.0,5.0,29.0,5.0,22,19.0,18.07,...,143.333333,106.333333,22.333333,166.333333,129.000000,61.666667,84.333333,-2.666667,117.333333,158.333333


In [59]:
# split data into train - test - oot
oot_pdf = data_pdf[(data_pdf['snapshot_date'] >= config["oot_start_date"].date()) & (data_pdf['snapshot_date'] <= config["oot_end_date"].date())]
train_test_pdf = data_pdf[(data_pdf['snapshot_date'] >= config["train_test_start_date"].date()) & (data_pdf['snapshot_date'] <= config["train_test_end_date"].date())]

feature_cols = [col for col in features_sdf.columns if col not in ['customer_id', 'snapshot_date']]

X_oot = oot_pdf[feature_cols]
y_oot = oot_pdf["label"]
X_train, X_test, y_train, y_test = train_test_split(
    train_test_pdf[feature_cols], train_test_pdf["label"], 
    test_size= 1 - config["train_test_ratio"],
    random_state=88,     # Ensures reproducibility
    shuffle=True,        # Shuffle the data before splitting
    stratify=train_test_pdf["label"]           # Stratify based on the label column
)


print('X_train', X_train.shape[0])
print('X_test', X_test.shape[0])
print('X_oot', X_oot.shape[0])
print('y_train', y_train.shape[0], round(y_train.mean(),2))
print('y_test', y_test.shape[0], round(y_test.mean(),2))
print('y_oot', y_oot.shape[0], round(y_oot.mean(),2))

X_train

X_train 4766
X_test 1192
X_oot 0
y_train 4766 0.28
y_test 1192 0.28
y_oot 0 nan


,age,annual_income,monthly_inhand_salary,num_bank_accounts,num_credit_card,interest_rate,num_of_loan,delay_from_due_date,num_of_delayed_payment,changed_credit_limit,...,avg_fe_11,avg_fe_12,avg_fe_13,avg_fe_14,avg_fe_15,avg_fe_16,avg_fe_17,avg_fe_18,avg_fe_19,avg_fe_20
153,39,10640.355469,1115.696289,6.0,3.0,13.0,2.0,12,18.0,9.620000,...,49.500000,78.250000,146.500000,63.500000,112.500000,68.500000,165.250000,193.000000,109.750000,65.250000
3658,36,57074.578125,4679.214844,7.0,9.0,15.0,3.0,23,25.0,2.790000,...,85.545455,98.363636,76.090909,141.545455,97.636364,119.636364,83.909091,82.727273,93.727273,112.363636
2977,34,100241.671875,8121.472656,8.0,5.0,10.0,4.0,20,14.0,1.800000,...,165.428571,114.428571,136.428571,53.142857,119.142857,91.571429,149.428571,142.714286,83.285714,99.428571
444,35,85742.820312,6400.101562,2.0,2.0,1.0,2.0,0,2.0,4.740000,...,136.000000,173.000000,-147.000000,-56.000000,243.000000,74.000000,195.000000,158.000000,115.000000,227.000000
3695,42,64270.121094,5500.843262,6.0,6.0,18.0,2.0,50,18.0,4.110000,...,133.916667,90.916667,103.166667,121.833333,111.583333,81.250000,100.583333,114.583333,169.750000,86.083333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1723,33,20762.970703,1651.247559,8.0,10.0,23.0,5.0,55,14.0,9.900000,...,155.500000,78.300000,42.600000,111.000000,144.800000,109.000000,120.400000,96.700000,123.300000,92.300000
3727,35,14252.750000,1434.729126,8.0,8.0,22.0,6.0,51,25.0,10.348008,...,129.166667,125.416667,107.333333,69.416667,146.250000,70.750000,99.666667,99.250000,120.000000,102.583333
1816,19,42870.210938,3523.517578,6.0,6.0,25.0,6.0,45,17.0,28.889999,...,115.444444,97.666667,52.888889,91.222222,133.555556,88.111111,102.444444,77.444444,158.222222,49.777778
5447,27,11465.919922,744.493347,7.0,5.0,11.0,0.0,24,6.0,7.350000,...,82.000000,-30.000000,123.500000,6.000000,93.000000,38.000000,136.500000,152.000000,63.000000,56.000000


## preprocess data

In [57]:
# set up standard scalar preprocessing
scaler = StandardScaler()

transformer_stdscaler = scaler.fit(X_train) # Q which should we use? train? test? oot? all?

# transform data
X_train_processed = transformer_stdscaler.transform(X_train)
X_test_processed = transformer_stdscaler.transform(X_test)
X_oot_processed = transformer_stdscaler.transform(X_oot)

print('X_train_processed', X_train_processed.shape[0])
print('X_test_processed', X_test_processed.shape[0])
print('X_oot_processed', X_oot_processed.shape[0])

pd.DataFrame(X_train_processed)

X_train_processed 4375
X_test_processed 1094
X_oot_processed 489


,0,1,2,3,4,5,6,7,8,9,...,62,63,64,65,66,67,68,69,70,71
0,0.388194,-0.058609,1.361163,-0.159237,-0.477608,-0.638764,-1.065005,-0.210142,0.262540,-0.463669,...,-0.237658,0.256885,-0.374717,-0.468449,0.649654,-0.524603,0.270467,0.972222,0.359031,-1.397426
1,-1.309023,-0.106641,-0.489943,0.600898,-0.738997,-0.198601,-0.643667,-0.957210,-0.874142,0.865838,...,0.602260,-1.554730,-0.597398,2.491774,1.499346,-0.278748,0.323075,-1.228767,0.558083,0.586794
2,0.765353,-0.098222,-0.223930,1.741100,-0.216220,-0.858845,-0.222328,-0.617634,-2.010823,0.126560,...,-2.012578,3.487909,0.834121,-0.810013,-1.411167,1.933941,-2.601305,-2.748278,-0.810784,0.308465
3,-0.648994,-0.116589,-0.940206,1.361033,0.829332,1.452008,-0.643667,2.506471,0.912072,-1.369881,...,0.725870,-3.693183,0.700512,1.353227,0.976215,0.265204,0.313791,0.060515,-0.480051,-1.271729
4,-1.120444,-0.105771,-0.552199,0.600898,-0.477608,0.021480,0.620349,-1.093041,-0.224610,1.393468,...,0.526192,0.452987,0.781632,-2.290125,0.162983,0.090033,-0.275727,-0.340083,0.447839,-0.311043
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4370,1.048223,-0.083933,0.276092,1.361033,1.090720,0.681724,-0.222328,1.283996,0.424923,0.114637,...,0.003224,1.396147,-0.062964,1.732742,-0.916570,1.417647,0.536601,-3.577102,0.640766,2.445317
4371,1.991122,-0.079825,0.405351,-0.539305,-0.216220,-0.418683,-0.643667,0.401096,-0.711759,-0.410012,...,1.657703,4.627171,-0.024790,-0.828989,-0.041513,-0.444700,-2.007145,-3.061389,-0.333058,-1.630864
4372,-1.403313,-0.026210,2.637907,-0.919372,-0.477608,0.241561,-0.222328,-0.413888,0.749689,-0.505403,...,-0.862049,-0.667599,-0.392213,-2.484627,0.267610,-3.182903,1.228240,0.949199,-0.254969,0.595772
4373,0.482484,-0.115046,-0.910066,0.600898,-0.738997,-0.858845,-0.222328,-1.025126,0.912072,0.585628,...,-0.681388,-0.890679,0.276359,-1.339227,-0.026718,0.409644,1.254544,0.781899,-0.239147,1.112528


In [38]:
cv = StratifiedKFold(n_splits=5)

## train model

In [39]:
# Define the XGBoost classifier
xgb_clf = xgb.XGBClassifier(eval_metric='logloss', random_state=88)

# Define the hyperparameter space to search
param_dist = {
    'n_estimators': [25, 50],
    'max_depth': [2, 3],  # lower max_depth to simplify the model
    'learning_rate': [0.01, 0.1],
    'subsample': [0.6, 0.8],
    'colsample_bytree': [0.6, 0.8],
    'gamma': [0, 0.1],
    'min_child_weight': [1, 3, 5],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

# Create a scorer based on AUC score
auc_scorer = make_scorer(roc_auc_score)

# Set up the random search with cross-validation
random_search = RandomizedSearchCV(
    estimator=xgb_clf,
    param_distributions=param_dist,
    scoring=auc_scorer,
    n_iter=100,  # Number of iterations for random search
    cv=3,       # Number of folds in cross-validation
    verbose=1,
    random_state=42,
    n_jobs=-1   # Use all available cores
)

# Perform the random search
random_search.fit(X_train_processed, y_train)

# Output the best parameters and best score
print("Best parameters found: ", random_search.best_params_)
print("Best AUC score: ", random_search.best_score_)

# Evaluate the model on the train set
best_model = random_search.best_estimator_
y_pred_proba = best_model.predict_proba(X_train_processed)[:, 1]
train_auc_score = roc_auc_score(y_train, y_pred_proba)
print("Train AUC score: ", train_auc_score)

# Evaluate the model on the test set
best_model = random_search.best_estimator_
y_pred_proba = best_model.predict_proba(X_test_processed)[:, 1]
test_auc_score = roc_auc_score(y_test, y_pred_proba)
print("Test AUC score: ", test_auc_score)

# Evaluate the model on the oot set
best_model = random_search.best_estimator_
y_pred_proba = best_model.predict_proba(X_oot_processed)[:, 1]
oot_auc_score = roc_auc_score(y_oot, y_pred_proba)
print("OOT AUC score: ", oot_auc_score)

print("TRAIN GINI score: ", round(2*train_auc_score-1,3))
print("Test GINI score: ", round(2*test_auc_score-1,3))
print("OOT GINI score: ", round(2*oot_auc_score-1,3))

Fitting 3 folds for each of 100 candidates, totalling 300 fits
Best parameters found:  {'subsample': 0.6, 'reg_lambda': 2, 'reg_alpha': 0.1, 'n_estimators': 50, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 0.6}
Best AUC score:  0.7751521753043905
Train AUC score:  0.9479223783612691
Test AUC score:  0.8765503530947717
OOT AUC score:  0.9210442488126954
TRAIN GINI score:  0.896
Test GINI score:  0.753
OOT GINI score:  0.842


In [40]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV

In [41]:
log_reg_model = LogisticRegression(random_state=88, solver='liblinear', C=1.0)

rfecv_log_reg = RFECV(
    estimator=log_reg_model,  # Use the pipeline here
    step=1,                      # Remove 1 feature at a time
    cv=cv,                       # Use our defined CV strategy
    scoring='accuracy',          # Score to optimize
    min_features_to_select=1,    # Minimum features to keep
    n_jobs=-1                    # Use all available cores
)
rfecv_log_reg.fit(X_train_processed, y_train)

# Evaluate the model on the train set
y_pred_proba = rfecv_log_reg.predict_proba(X_train_processed)[:, 1]
train_auc_score = roc_auc_score(y_train, y_pred_proba)
print("Train AUC score: ", train_auc_score)

# Evaluate the model on the test set
y_pred_proba = rfecv_log_reg.predict_proba(X_test_processed)[:, 1]
test_auc_score = roc_auc_score(y_test, y_pred_proba)
print("Test AUC score: ", test_auc_score)

# Evaluate the model on the oot set_
y_pred_proba = rfecv_log_reg.predict_proba(X_oot_processed)[:, 1]
oot_auc_score = roc_auc_score(y_oot, y_pred_proba)
print("OOT AUC score: ", oot_auc_score)

print("TRAIN GINI score: ", round(2*train_auc_score-1,3))
print("Test GINI score: ", round(2*test_auc_score-1,3))
print("OOT GINI score: ", round(2*oot_auc_score-1,3))

Train AUC score:  0.9101607265042799
Test AUC score:  0.8771141178565071
OOT AUC score:  0.9227817676358161
TRAIN GINI score:  0.82
Test GINI score:  0.754
OOT GINI score:  0.846


In [31]:
logreg_cv = LogisticRegressionCV(
    Cs=10,                      # Test 10 C values (inverse regularization strength)
    cv=5,                       # 5-fold cross-validation
    penalty='l2',               # L2 regularization (Ridge)
    solver='liblinear',         # Solver that supports L2 penalty
    scoring='accuracy',         # Metric used to find the best C
    random_state=88,
    max_iter=1000               # Increase max_iter for convergence stability
)

# Fitting the model automatically performs cross-validation 
# and selects the best C, then refits the model on the entire training set.
logreg_cv.fit(X_train_processed, y_train)

# 5. Get the Optimal Regularization Parameter (C)
# C_ is an array, where each element corresponds to the best C found 
# for each class/target. For binary classification, it has one element.
optimal_C = logreg_cv.C_[0] 
optimal_lambda = 1 / optimal_C # Regularization strength (lambda) is 1/C

# Evaluate the model on the train set
y_pred_proba = logreg_cv.predict_proba(X_train_processed)[:, 1]
train_auc_score = roc_auc_score(y_train, y_pred_proba)
print("Train AUC score: ", train_auc_score)

# Evaluate the model on the test set
y_pred_proba = logreg_cv.predict_proba(X_test_processed)[:, 1]
test_auc_score = roc_auc_score(y_test, y_pred_proba)
print("Test AUC score: ", test_auc_score)

# Evaluate the model on the oot set_
y_pred_proba = logreg_cv.predict_proba(X_oot_processed)[:, 1]
oot_auc_score = roc_auc_score(y_oot, y_pred_proba)
print("OOT AUC score: ", oot_auc_score)

print("TRAIN GINI score: ", round(2*train_auc_score-1,3))
print("Test GINI score: ", round(2*test_auc_score-1,3))
print("OOT GINI score: ", round(2*oot_auc_score-1,3))

Train AUC score:  0.6536371471903986
Test AUC score:  0.6521595779756016
OOT AUC score:  0.6289142437931967
TRAIN GINI score:  0.307
Test GINI score:  0.304
OOT GINI score:  0.258


## prepare model artefact to save

In [19]:
model_artefact = {}

model_artefact['model'] = best_model
model_artefact['model_version'] = "credit_model_"+config["model_train_date_str"].replace('-','_')
model_artefact['preprocessing_transformers'] = {}
model_artefact['preprocessing_transformers']['stdscaler'] = transformer_stdscaler
model_artefact['data_dates'] = config
model_artefact['data_stats'] = {}
model_artefact['data_stats']['X_train'] = X_train.shape[0]
model_artefact['data_stats']['X_test'] = X_test.shape[0]
model_artefact['data_stats']['X_oot'] = X_oot.shape[0]
model_artefact['data_stats']['y_train'] = round(y_train.mean(),2)
model_artefact['data_stats']['y_test'] = round(y_test.mean(),2)
model_artefact['data_stats']['y_oot'] = round(y_oot.mean(),2)
model_artefact['results'] = {}
model_artefact['results']['auc_train'] = train_auc_score
model_artefact['results']['auc_test'] = test_auc_score
model_artefact['results']['auc_oot'] = oot_auc_score
model_artefact['results']['gini_train'] = round(2*train_auc_score-1,3)
model_artefact['results']['gini_test'] = round(2*test_auc_score-1,3)
model_artefact['results']['gini_oot'] = round(2*oot_auc_score-1,3)
model_artefact['hp_params'] = random_search.best_params_


pprint.pprint(model_artefact)

{'data_dates': {'model_train_date': datetime.datetime(2024, 9, 1, 0, 0),
                'model_train_date_str': '2024-09-01',
                'oot_end_date': datetime.datetime(2024, 8, 31, 0, 0),
                'oot_period_months': 2,
                'oot_start_date': datetime.datetime(2024, 7, 1, 0, 0),
                'train_test_end_date': datetime.datetime(2024, 6, 30, 0, 0),
                'train_test_period_months': 12,
                'train_test_ratio': 0.8,
                'train_test_start_date': datetime.datetime(2023, 7, 1, 0, 0)},
 'data_stats': {'X_oot': 1003,
                'X_test': 1192,
                'X_train': 4766,
                'y_oot': np.float64(0.29),
                'y_test': np.float64(0.28),
                'y_train': np.float64(0.28)},
 'hp_params': {'colsample_bytree': 0.8,
               'gamma': 0.1,
               'learning_rate': 0.1,
               'max_depth': 3,
               'min_child_weight': 5,
               'n_estimators': 50,
        

## save artefact to model bank

In [20]:
# create model_bank dir
model_bank_directory = "model_bank/"

if not os.path.exists(model_bank_directory):
    os.makedirs(model_bank_directory)

In [21]:
# Full path to the file
file_path = os.path.join(model_bank_directory, model_artefact['model_version'] + '.pkl')

# Write the model to a pickle file
with open(file_path, 'wb') as file:
    pickle.dump(model_artefact, file)

print(f"Model saved to {file_path}")


Model saved to model_bank/credit_model_2024_09_01.pkl


## test load pickle and make model inference

In [22]:
# Load the model from the pickle file
with open(file_path, 'rb') as file:
    loaded_model_artefact = pickle.load(file)

y_pred_proba = loaded_model_artefact['model'].predict_proba(X_oot_processed)[:, 1]
oot_auc_score = roc_auc_score(y_oot, y_pred_proba)
print("OOT AUC score: ", oot_auc_score)

print("Model loaded successfully!")

OOT AUC score:  0.6264093208231979
Model loaded successfully!
